In [5]:
import os

print(os.getcwd())
print(os.listdir())

/Users/saraelbachouti/Desktop/UCV-Churn
['eda5.ipynb', 'primermodelosimple.ipynb', '.DS_Store', 'LICENSE', 'requirements.txt', 'tercermodelo.ipynb', 'modelomasexacto.ipynb', 'config', 'Dockerfile', 'eda4.ipynb', 'edatercerenfoque.ipynb', 'models', 'docs', 'README.md', 'cuartomodelo.ipynb', '.gitignore', 'edasegundoenfoque.ipynb', 'dataset_limpio_churn.csv', '.git', 'processed', 'data', 'edaprimerenfoque.ipynb', 'notebooks', 'reports', 'src']


In [5]:
# ============================================================
# MODELO FINAL XGBOOST CORRECTO
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ============================================================
# 1. CARGAR DATASET
# ============================================================

df_model = pd.read_csv("dataset_limpio_churn.csv")

dataset = df_model.copy()

print("Shape inicial:", dataset.shape)


# ============================================================
# 2. FECHAS
# ============================================================

if "fecha" in dataset.columns:

    dataset["fecha"] = pd.to_datetime(
        dataset["fecha"],
        errors="coerce"
    )

    if "cliente_id" in dataset.columns:

        dataset = dataset.sort_values(
            ["cliente_id", "fecha"]
        )


# ============================================================
# 3. FEATURES TEMPORALES
# ============================================================

if (
    "cliente_id" in dataset.columns
    and
    "importe_total" in dataset.columns
):

    dataset["media_importe_3m"] = (
        dataset
        .groupby("cliente_id")["importe_total"]
        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).mean()
        )
    )

    dataset["importe_mes_anterior"] = (
        dataset
        .groupby("cliente_id")
        ["importe_total"]
        .shift(1)
    )

    dataset["subida_factura"] = (
        dataset["importe_total"]
        -
        dataset["importe_mes_anterior"]
    )


if (
    "cliente_id" in dataset.columns
    and
    "dias_retraso_pago" in dataset.columns
):

    dataset["media_retraso_3m"] = (

        dataset

        .groupby("cliente_id")

        ["dias_retraso_pago"]

        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).mean()
        )
    )


if (
    "cliente_id" in dataset.columns
    and
    "impago_flag" in dataset.columns
):

    dataset["impagos_3m"] = (

        dataset

        .groupby("cliente_id")

        ["impago_flag"]

        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).sum()
        )
    )


# ============================================================
# 4. VARIABLES INTERACCIÓN
# ============================================================

if (
    "dias_retraso_pago" in dataset.columns
    and
    "importe_total" in dataset.columns
):

    dataset["riesgo_pago"] = (
        dataset["dias_retraso_pago"]
        *
        dataset["importe_total"]
    )


if (
    "impago_flag" in dataset.columns
    and
    "stress_calidad_lag" in dataset.columns
):

    dataset["riesgo_finanzas"] = (
        dataset["impago_flag"]
        *
        dataset["stress_calidad_lag"]
    )


# ============================================================
# 5. ELIMINAR COLUMNAS
# ============================================================

eliminar = [

    "cliente_id",
    "fecha",
    "zona_id",
    "zona_id_x",
    "zona_id_y"

]

dataset = dataset.drop(
    columns=[
        c
        for c
        in eliminar
        if c
        in dataset.columns
    ],
    errors="ignore"
)


# ============================================================
# 6. TARGET
# ============================================================

if "churn" not in dataset.columns:

    raise Exception(
        "No existe columna churn"
    )


X = dataset.drop(
    columns="churn"
)

y = dataset["churn"]


# ============================================================
# 7. VARIABLES CATEGÓRICAS
# ============================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ============================================================
# 8. LIMPIEZA
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(
        numeric_only=True
    )
)

X = X.fillna(0)


# ============================================================
# 9. TRAIN TEST
# ============================================================

X_train, X_test, y_train, y_test = (

    train_test_split(

        X,
        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

)


# ============================================================
# 10. BALANCEO
# ============================================================

peso = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

print(
    "scale_pos_weight:",
    round(peso,2)
)


# ============================================================
# 11. MODELO
# ============================================================

modelo = XGBClassifier(

    n_estimators=500,

    learning_rate=0.05,

    max_depth=4,

    min_child_weight=10,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=peso,

    random_state=42,

    eval_metric="logloss",

    n_jobs=-1

)


modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 12. PREDICCIÓN
# ============================================================

proba = (
    modelo
    .predict_proba(
        X_test
    )[:,1]
)


threshold = 0.60


pred = (
    proba
    >=
    threshold
).astype(int)


# ============================================================
# RESULTADOS
# ============================================================

print("\nRESULTADOS")

print(
"Accuracy:",
round(
accuracy_score(
y_test,
pred
)*100,
2
)
)

print(
"Precision:",
round(
precision_score(
y_test,
pred,
zero_division=0
),
4
)
)

print(
"Recall:",
round(
recall_score(
y_test,
pred
),
4
)
)

print(
"F1:",
round(
f1_score(
y_test,
pred
),
4
)
)

print(
"ROC:",
round(
roc_auc_score(
y_test,
proba
),
4
)
)

print(
"PR:",
round(
average_precision_score(
y_test,
proba
),
4
)
)

print(
"\nMatriz:"
)

print(
confusion_matrix(
y_test,
pred
)
)

print(
"\nClassification Report"
)

print(
classification_report(
y_test,
pred
)
)

Shape inicial: (317579, 35)
scale_pos_weight: 160.41

RESULTADOS
Accuracy: 90.26
Precision: 0.0213
Recall: 0.3274
F1: 0.04
ROC: 0.7078
PR: 0.0237

Matriz:
[[57200  5922]
 [  265   129]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.91      0.95     63122
           1       0.02      0.33      0.04       394

    accuracy                           0.90     63516
   macro avg       0.51      0.62      0.49     63516
weighted avg       0.99      0.90      0.94     63516



In [7]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN DEFENDIBLE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)


# ======================================================
# 2. CREAR VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso"] = np.where(
    df["dias_retraso_pago"] > 0,
    1,
    0
)

df["riesgo_financiero"] = (
    df["impago_flag"].fillna(0) * 5
    + df["dias_retraso_pago"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
)

df["retraso_impago"] = (
    df["dias_retraso_pago"].fillna(0)
    * df["impago_flag"].fillna(0)
)


# ======================================================
# 3. ELIMINAR VARIABLES NO VÁLIDAS
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

df = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 4. TARGET
# ======================================================

X = df.drop(columns="churn")
y = df["churn"]


# ======================================================
# 5. VARIABLES CATEGÓRICAS
# ======================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ======================================================
# 6. LIMPIEZA
# ======================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)


# ======================================================
# 7. TRAIN / TEST
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.20,
    random_state=42
)


# ======================================================
# 8. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight:", round(peso, 2))


# ======================================================
# 9. MODELO XGBOOST
# ======================================================

modelo = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=20,
    gamma=5,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=5,
    reg_lambda=10,
    scale_pos_weight=peso,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train, y_train)


# ======================================================
# 10. PROBABILIDADES
# ======================================================

proba = modelo.predict_proba(X_test)[:, 1]


# ======================================================
# 11. TABLA DE THRESHOLDS
# ======================================================

resultados = []

for threshold in np.arange(0.30, 0.81, 0.01):

    pred_temp = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        pred_temp
    ).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_test, pred_temp),
        "precision": precision_score(y_test, pred_temp, zero_division=0),
        "recall": recall_score(y_test, pred_temp, zero_division=0),
        "f1": f1_score(y_test, pred_temp, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)


print("\n" + "="*70)
print("TABLA DE THRESHOLDS ORDENADA POR F1")
print("="*70)

print(
    df_thresholds
    .sort_values(by="f1", ascending=False)
    .head(15)
)


print("\n" + "="*70)
print("TABLA DE THRESHOLDS PARA DETECTAR MÁS CHURN")
print("="*70)

print(
    df_thresholds
    .sort_values(by="clientes_churn_detectados", ascending=False)
    .head(15)
)


# ======================================================
# 12. ELEGIR THRESHOLD FINAL
# ======================================================
# Puedes cambiar este valor:
# 0.70 = más equilibrado
# 0.60 = detecta más churn, pero aumenta falsos positivos
# 0.55 = detecta todavía más, pero con más ruido

threshold_final = 0.60

pred = (proba >= threshold_final).astype(int)


# ======================================================
# 13. RESULTADOS FINALES
# ======================================================

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred
).ravel()

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)
roc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)


print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred, zero_division=0))


# ======================================================
# 14. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)

print(importancias.head(20))

Shape inicial: (317579, 35)

scale_pos_weight: 160.41

TABLA DE THRESHOLDS ORDENADA POR F1
    threshold  accuracy  precision    recall        f1  \
46       0.76  0.966355   0.031200  0.147208  0.051487   
45       0.75  0.961868   0.028372  0.154822  0.047956   
49       0.79  0.976352   0.031303  0.093909  0.046954   
48       0.78  0.973125   0.030064  0.106599  0.046901   
47       0.77  0.969882   0.029138  0.119289  0.046836   
44       0.74  0.957381   0.026994  0.167513  0.046495   
43       0.73  0.952658   0.026459  0.185279  0.046305   
50       0.80  0.979123   0.032129  0.081218  0.046043   
42       0.72  0.948234   0.025262  0.195431  0.044741   
41       0.71  0.942991   0.024462  0.210660  0.043834   
40       0.70  0.937858   0.023854  0.225888  0.043152   
39       0.69  0.932836   0.023387  0.241117  0.042639   
38       0.68  0.926680   0.022192  0.251269  0.040783   
36       0.66  0.914982   0.021232  0.281726  0.039488   
37       0.67  0.920603   0.021215  0.2